# 1、content的使用

举例1：保存多模态的数据，使用字典列表

In [3]:
from langchain_deepseek import ChatDeepSeek
import base64
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

# load_dotenv(override=True)
#
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    api_base=os.getenv("DEEPSEEK_BASE_URL"),
)

def encode_image(img_path, img_type='jpeg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:  # rb 表示二进制只读方式打开文件，读图必须用 rb
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode('utf-8')}"  # 内层改为单引号，python 3.12+ 才支持内外同引号
        # img_file.read()
        ## .read() 不带参数 → 一次性读到文件末尾。返回值类型是 bytes（字节串）
        # b64encode() 接收 bytes，返回 Base64 编码后的 bytes
        ## 为什么需要它？因为要把二进制图片塞进纯文本的 HTTP JSON 请求里，Base64 能把任意二进制安全地编码成 A-Z a-z 0-9 + / = 这些可打印字符。
        # bytes.decode(encoding) → 把字节串按指定编码解码成 str。
        ## Base64 的输出只含 ASCII 字符，所以用 'utf-8'（或 'ascii'）都能正确解码。结果就是纯字符串。

# 图像路径
img_path = "image_test.png"
# 获取图像base64编码字符串
base64_image = encode_image(img_path, img_type='png')

response = model.invoke(
    [
        HumanMessage(
            content=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image_url',
                    # "image_url": base64_image,
                    'image_url': {'url': base64_image},     # 正确写法：包一层 dict
                }
            ]
        )
    ]
)
print(response.content)

这张图里主要展示的是一瓶**化妆品**，确切地说，是一瓶**粉底液**。

具体细节如下：

*   **产品外观**：瓶身是透明方形的玻璃材质，里面装着浅米色（粉底液常见色号）的浓稠液体。瓶盖是透明亚克力材质，内部有金色的按压泵头。
*   **标签设计**：瓶身正面设有一个醒目的金色长方形金属标牌（带有品牌英文字母或标识），下方还有一个较小的带有黑色字体的长方形标签。瓶底也有黑色的产品英文说明。
*   **光影与背景**：产品被放置在一个米黄色/沙色的平滑背景上。侧面有较硬的光线照射，在瓶身的左后方投射出清晰的阴影，整体营造出一种高端、干净、极简的广告产品摄影风格。


# 2、content_blocks的使用

举例1 ：输入的格式化

In [4]:
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    api_base=os.getenv("DEEPSEEK_BASE_URL"),
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        # 此种格式可用
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': '这张图里有什么？'},
        #         {
        #             'type': 'image_url',
        #             'image_url': {'url': base64_image},
        #         }
        #     ]
        # )
        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/png',
                }
            ]
        )

    ]
)
print(response.content)

这张图里是一瓶 Tom Ford（汤姆·福特）的粉底液。它有一个长方形的透明玻璃瓶身，里面装着米色的液体，顶部配有金色的按压泵头。瓶身上方贴有金色的品牌标识，下方的黑色小字写着“SKIN ILLUMINATING FOUNDATION”和“SOFT RADIANT FOUNDATION”。背景是温暖的米色，光线打在产品上，在左下方投射出清晰的阴影，整体呈现出一张商业产品摄影的风格。


作为对比

In [5]:
import base64
from langchain.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv(override=True)

model = init_chat_model(
    model="claude-haiku-4-5",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        # 传统的写法：使用content。读取失败
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': '这张图里有什么？'},
        #         {
        #             'type': 'image_url',
        #             "image_url": base64_image,
        #         }
        #     ]
        # )

        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/png',
                }
            ]
        )
    ]
)
print(response.content)

这张图里是一瓶**香水或液体化妆品**。具体特点包括：

- **瓶身**：透明玻璃瓶，呈方形或矩形设计
- **颜色**：液体呈浅金色或米色
- **瓶盖**：金色的喷雾泵头或压泵设计
- **风格**：高级、精致的美妆产品包装
- **背景**：米色/奶油色背景，光线充足

从外观推测，这可能是：
- 香水
- 粉底液
- 妆前乳或定妆喷雾
- 其他高端护肤/彩妆产品

整个画面呈现出专业的产品摄影风格，强调产品的质感和高级感。


举例2：输出的格式化

In [5]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={"thinking": {"type": "enabled"}},
)

response = model.invoke("你好，一句话回答")
print('=' * 20, '-> response <-', '=' * 20)
print(response)
print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)
print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)

==================== -> response <- ====================
content='你好，请告诉我你需要什么帮助。' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户。用户说“你好，一句话回答”。可能要求用一句话回答。我们需要回应。应该只是打招呼？需要一句话。可以：“你好！有什么我可以帮你的吗？” 这是一句话。注意用户要求“一句话回答”，所以只给一句话。不要多余。用中文。可以回答“你好，请告诉我你需要什么帮助。”这是一句。确保一句话。最终。 '} response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 34, 'total_tokens': 123, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 80, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 34}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '01a8c276-e284-4202-be71-e94e8650008f', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a11f51-0ff4-7702-9781-b9790ff53709-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_token